# 5장 — 다수결 회의: self-consistency

웹 챕터(5장)와 **같은 예제, 같은 수치**를 파이썬으로 직접 계산합니다: 이항 다수결의 힘,
p<50% 역전, 상관된 오류의 함정까지. 난수 없는 완전 해석적 계산이라 결과가 항상 같습니다.

> 🤖 **AI 튜터와 함께 공부하세요**: 아래 셀을 실행하면 화면 오른쪽 아래에 튜터 버튼이 뜹니다.
> 로컬 Ollama 가 꺼져 있어도 노트북 학습에는 지장이 없습니다.

In [ ]:
from tutor import chat
chat('ch5')  # 이 챕터를 아는 AI 튜터 (로컬 Ollama — 없어도 학습엔 지장 없음)

## 1. 이항 다수결 공식

정답률 p짜리 **독립인** 표 k개(홀수) 중 과반이 정답일 확률. 3부 5장의 이항분포 합입니다:

$$P(\text{다수결 정답}) = \sum_{i=\lceil k/2 \rceil}^{k} \binom{k}{i} p^i (1-p)^{k-i}$$

In [ ]:
import math

def majority(p, k):
    """독립인 표 k개(홀수) 중 과반이 정답일 확률 — 이항 합. 웹 VoteLab 과 동일 공식."""
    m = k // 2 + 1
    return sum(math.comb(k, i) * p**i * (1 - p)**(k - i) for i in range(m, k + 1))

for k in [1, 3, 5, 9, 15]:
    print(f'p=70%, {k:>2}표 다수결 → {majority(0.7, k)*100:.1f}%')

# 웹 실험실·본문 수치와 일치하는지 자가검증
assert abs(majority(0.7, 3) - 0.78400) < 1e-5   # StepQuiz: 0.441 + 0.343
assert abs(majority(0.7, 5) - 0.83692) < 1e-5
assert abs(majority(0.7, 9) - 0.90119) < 1e-5
assert abs(majority(0.7, 15) - 0.94999) < 1e-5
print('✅ 다수결의 힘: 70% 한 표 → 5표 83.7% → 15표 95.0%')

## 2. 두 가지 배신: p<50% 역전, 상관된 오류

① 한 표가 50% 미만이면 과반은 **틀린 답**에 모입니다. 다수결이 악화 장치가 돼요.

② 표끼리 닮으면(상관, 기호 ρ, 읽기: **로**. 표들이 얼마나 닮았는지의 다이얼로 0 = 독립,
1 = 전원이 한 표를 복사) 유효 표 수가 줄어듭니다. 간단 모형: 확률 ρ로 전원이 한 표를 복사,
확률 1−ρ로 독립 투표 → $P = \rho \cdot p + (1-\rho) \cdot P_{\text{독립}}$.

In [ ]:
# ① 역전 — 40%짜리 표 5개의 다수결
rev = majority(0.4, 5)
print(f'p=40%, 5표 다수결 → {rev*100:.1f}%  (한 표 40%보다 나빠짐!)')
assert abs(rev - 0.31744) < 1e-5

# ② 상관 — ρ=0.5, p=70%, 5표
def majority_corr(p, k, rho):
    return rho * p + (1 - rho) * majority(p, k)

corr = majority_corr(0.7, 5, 0.5)
print(f'ρ=0.5, p=70%, 5표 → {corr*100:.1f}%  (독립이면 {majority(0.7,5)*100:.1f}%)')
assert abs(corr - 0.76846) < 1e-5
assert abs(majority_corr(0.7, 5, 1.0) - 0.7) < 1e-12  # 완전 복사 = 그냥 한 표
print('✅ 역전·상관 수치 자가검증 통과. 회의는 머릿수가 아니라 독립적인 과반이 만든다')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

ks = [1, 3, 5, 7, 9, 11, 13, 15]
fig, ax = plt.subplots(figsize=(8, 4.5))
for p, color in [(0.9, '#10b981'), (0.7, '#6366f1'), (0.55, '#f59e0b')]:
    ax.plot(ks, [majority(p, k) * 100 for k in ks], marker='o', ms=4, color=color, label=f'한 표 {int(p*100)}%')
ax.plot(ks, [majority(0.45, k) * 100 for k in ks], marker='x', ms=5, ls='--', color='#f43f5e', label='한 표 45% (역전!)')
ax.axhline(50, ls='--', lw=1, color='#94a3b8')
ax.set_xlabel('투표자 수 k (홀수)')
ax.set_ylabel('다수결 정답률 (%)')
ax.set_title('이항 다수결: p>50%면 증폭, p<50%면 악화 (콩도르세 배심원 정리)')
ax.set_xticks(ks)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 상관 스윕 — ρ가 커질수록 다수결의 이득이 녹아내린다 (p=0.7, 5표)
rhos = [i / 10 for i in range(11)]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(rhos, [majority_corr(0.7, 5, r) * 100 for r in rhos], marker='o', ms=4, color='#f43f5e')
ax.axhline(70, ls='--', lw=1, color='#94a3b8')
ax.annotate('한 표의 정답률 70%', (0.55, 70.6), color='#64748b', fontsize=9)
ax.set_xlabel('상관 ρ (0 = 독립, 1 = 전원이 한 표를 복사)')
ax.set_ylabel('다수결 정답률 (%)')
ax.set_title('상관된 오류의 함정: 닮은 표는 머릿수가 아니다 (p=70%, 5표)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. 오늘의 발견

- **다수결은 확률 증폭기**: 70% 한 표가 5표에서 83.7%, 15표에서 95.0% (이항 합).
- **전제 ①**: p > 50%. 미만이면 틀린 답으로 수렴한다 (40%·5표 → 31.7%).
- **전제 ②**: 표의 독립. 같은 모델의 표는 닮아서(상관된 오류) 이론만큼 안 오른다 (ρ=0.5면 76.8%).
- 답이 하나로 수렴하는 문제(수학·객관식·코드 테스트)에서만 강력. 열린 과업은 6장 검수자의 몫.

다음 장(6장 검수자)에서 되살리는 세 번째 길, "검수 통과한 답은 얼마나 믿을 수 있나"를 베이즈로 계산합니다.
**웹 챕터로 돌아가 연습 10문제를 풀면 진도가 채워져요.**